# AgencyOS — Phase 3 GPU and storage preflight

This is a non-destructive check. It does not download models or datasets, call AI APIs, or train.

Keep this notebook private. Run with a GPU enabled. The scratch-storage canary is temporary and is deleted immediately.

A local write test does not prove persistence across sessions; restart the notebook and verify a private checkpoint location before Phase 3 sign-off.

In [ ]:
import os, platform, shutil, sys, tempfile
from pathlib import Path
print('AgencyOS Phase 3 preflight')
print('Python:', platform.python_version())
if sys.version_info < (3, 10): raise RuntimeError('Python 3.10 or newer is required.')
import torch
print('PyTorch:', torch.__version__, '| CUDA build:', torch.version.cuda)
if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU unavailable. Enable GPU in notebook settings.')
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i); gib=p.total_memory/(1024**3)
    print(f'GPU {i}: {p.name} | {gib:.2f} GiB | capability {p.major}.{p.minor}')
    if gib < 12: raise RuntimeError(f'GPU {i} has less than 12 GiB VRAM.')
probe=torch.ones((1024,1024),dtype=torch.float16,device='cuda:0')
print('CUDA allocation/release: PASS | checksum',float(probe.sum().item()))
del probe; torch.cuda.synchronize(); torch.cuda.empty_cache()
try:
    import psutil
    m=psutil.virtual_memory(); print(f'RAM: {m.total/(1024**3):.2f} GiB total | {m.available/(1024**3):.2f} GiB available')
except ImportError: print('RAM details unavailable; psutil is optional.')
target=Path(os.environ.get('AGENCYOS_CHECKPOINT_DIR','/kaggle/working')); target.mkdir(parents=True,exist_ok=True)
d=shutil.disk_usage(target); print(f'Scratch path: {target} | free: {d.free/(1024**3):.2f} GiB')
with tempfile.NamedTemporaryFile(prefix='agencyos-check-',dir=target,delete=True) as f: f.write(b'phase3-check'); f.flush()
print('Temporary storage write/delete: PASS')
print('RESULT: PASS — local hardware and scratch checks completed.')
print('PERSISTENCE GATE: still pending. Restart session and verify private checkpoint read-back before sign-off.')